# An object refusal can explain a business rule

An order already shipped cannot be cancelled. The client needs a stable reason to display the appropriate message.

Run with the Python environment of this checkout (`.venv`). These examples use the API in this branch; installing a released package may give a different API.

[Script](09_reasons.py) · [Tutorial](../../docs/tutorials/step-03-authorization-and-roles.md)

## Imports

In [1]:
"""An object refusal can explain a business rule."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Refused, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

## The rule

An order already shipped cannot be cancelled. The client needs a stable reason to display the appropriate message.

In [2]:
class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


STATUS = {"ord-001": "shipped"}


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @access_decide("Check whether the order can be cancelled")
    async def cancel_access_decide(
        self,
        params: OrderParams,
        box: ToolsBox,
        connections: dict[str, BaseResource],
    ) -> Verdict:
        """Decide access before cancelling the order."""
        if STATUS[params.order_id] == "shipped":
            return Refused("ORDER_ALREADY_SHIPPED")
        return Allowed()

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params, state, box, connections):
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)

## Try it

Run the cells in order. The calls below are the same as in the script.

In [3]:
machine = ActionProductMachine(cache_coordinator=None)
caller = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))

In [4]:
answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="ord-001"))
print(answer.model_dump(mode="json"))

{'kind': 'refused', 'gate': 'ACCESS_DECIDE', 'reason': 'ORDER_ALREADY_SHIPPED'}


## What the result means

Refused defaults to ACCESS_DECIDE. Its reason is exactly the application-provided string. In an application that restricts object visibility, establish ownership before disclosing an order status.